In [ ]:
import pandas as pd
import re
import numpy as np
import string
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

In [ ]:
df=pd.read_csv('/content/sentiment_dataset_raw.csv')
df

,Review,sentiment
0,I absolutely loved this movie! 😊 😍,positive
1,"Terrible experience, would not recommend. !!",negative
2,I absolutely loved this movie! 😊 😍,positive
3,"Had a great time with friends, everything was ...",positive
4,I absolutely loved this movie! 😊,positive
...,...,...
5995,What a fantastic experience. Will come again! !!!,positive
5996,Worst product ever. Total waste of money!,negative
5997,"Had a great time with friends, everything was ...",positive
5998,"The service was excellent, highly recommend!",positive


In [ ]:
#Lowercasing : all charac all convert to lowercase
df['Review']=df['Review'].str.lower()
df

,Review,sentiment
0,i absolutely loved this movie! 😊 😍,positive
1,"terrible experience, would not recommend. !!",negative
2,i absolutely loved this movie! 😊 😍,positive
3,"had a great time with friends, everything was ...",positive
4,i absolutely loved this movie! 😊,positive
...,...,...
5995,what a fantastic experience. will come again! !!!,positive
5996,worst product ever. total waste of money!,negative
5997,"had a great time with friends, everything was ...",positive
5998,"the service was excellent, highly recommend!",positive


remove HTML tags

In [ ]:
df['Review'].str.contains('<.*?>').value_counts()

,count
Review,
False,5126
True,874


In [ ]:
df

,Review,sentiment
0,i absolutely loved this movie! 😊 😍,positive
1,"terrible experience, would not recommend. !!",negative
2,i absolutely loved this movie! 😊 😍,positive
3,"had a great time with friends, everything was ...",positive
4,i absolutely loved this movie! 😊,positive
...,...,...
5995,what a fantastic experience. will come again! !!!,positive
5996,worst product ever. total waste of money!,negative
5997,"had a great time with friends, everything was ...",positive
5998,"the service was excellent, highly recommend!",positive


In [ ]:
a=lambda x: BeautifulSoup(x,'html.parser').get_text()
a('terrible experience, would not recommend. <br>')

'terrible experience, would not recommend. '

In [ ]:
df['Review']=df['Review'].apply(lambda x: BeautifulSoup(x, 'html.parser').get_text())
df

,Review,sentiment
0,i absolutely loved this movie! 😊 😍,positive
1,"terrible experience, would not recommend. !!",negative
2,i absolutely loved this movie! 😊 😍,positive
3,"had a great time with friends, everything was ...",positive
4,i absolutely loved this movie! 😊,positive
...,...,...
5995,what a fantastic experience. will come again! !!!,positive
5996,worst product ever. total waste of money!,negative
5997,"had a great time with friends, everything was ...",positive
5998,"the service was excellent, highly recommend!",positive


In [ ]:
df[df['Review'].str.contains('<.*?>')]

,Review,sentiment


REMOVE URL'S

In [ ]:
df[df['Review'].str.contains('http')]

,Review,sentiment
5,"the service was excellent, highly recommend! h...",positive
14,"superb product quality, very satisfied. http:/...",positive
24,i hated every minute of it. 😡 http://example.com,negative
41,"superb product quality, very satisfied. http:/...",positive
50,not worth the time or the price. http://exampl...,negative
...,...,...
5959,i hated every minute of it. 😡 http://example.com,negative
5978,"had a great time with friends, everything was ...",positive
5980,i'm very disappointed with the service. http:/...,negative
5981,i hated every minute of it. 😡 http://example.com,negative


In [ ]:
df['Review']=df['Review'].apply(lambda x: re.sub(r'http\S+','',x))


In [ ]:
df[df['Review'].str.contains('http')]

,Review,sentiment


REMOVE PUNCTUATION MARKS

In [ ]:
df['Review']=df['Review'].apply(lambda x: x.translate(str.maketrans('','',string.punctuation)))
df['Review']

,Review
0,i absolutely loved this movie 😊 😍
1,terrible experience would not recommend
2,i absolutely loved this movie 😊 😍
3,had a great time with friends everything was p...
4,i absolutely loved this movie 😊
...,...
5995,what a fantastic experience will come again
5996,worst product ever total waste of money
5997,had a great time with friends everything was p...
5998,the service was excellent highly recommend


In [ ]:
word_data={
    'ur':'your',
    'gm':'good morning',
    'idk':'i do not know',
    'eod':'end of the day',
    'gn':'good night',
    'lol':'laught out loud',
    'temp':'temperature',
    'eta':'estimated time arrival'
}

In [ ]:
def expand_chat_words(text):
  return ' '.join([word_data[word] if word in word_data else word for word in text.split()])

In [ ]:
df['Review']=df['Review'].apply(expand_chat_words)

#Handling emojis :
       convert emotion images into text
       purpose ---> emojis carry emotional value

In [ ]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 39.9 MB/s eta 0:00:00


In [ ]:
import emoji
emoji.demojize('😀',delimiters=("",""))

'grinning_face'

In [ ]:
df

,Review,sentiment
0,i absolutely loved this movie 😊 😍,positive
1,terrible experience would not recommend,negative
2,i absolutely loved this movie 😊 😍,positive
3,had a great time with friends everything was p...,positive
4,i absolutely loved this movie 😊,positive
...,...,...
5995,what a fantastic experience will come again,positive
5996,worst product ever total waste of money,negative
5997,had a great time with friends everything was p...,positive
5998,the service was excellent highly recommend,positive


In [ ]:
df['cleaned_data']=df['Review'].apply(lambda x: emoji.demojize(x,delimiters=(" "," ")))

In [ ]:
df

,Review,sentiment,cleaned_data
0,i absolutely loved this movie 😊 😍,positive,i absolutely loved this movie smiling_face_wi...
1,terrible experience would not recommend,negative,terrible experience would not recommend
2,i absolutely loved this movie 😊 😍,positive,i absolutely loved this movie smiling_face_wi...
3,had a great time with friends everything was p...,positive,had a great time with friends everything was p...
4,i absolutely loved this movie 😊,positive,i absolutely loved this movie smiling_face_wi...
...,...,...,...
5995,what a fantastic experience will come again,positive,what a fantastic experience will come again
5996,worst product ever total waste of money,negative,worst product ever total waste of money
5997,had a great time with friends everything was p...,positive,had a great time with friends everything was p...
5998,the service was excellent highly recommend,positive,the service was excellent highly recommend


In [ ]:
#remove stop words
import nltk
nltk.download('stopwords')
stop_words=set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
len(stop_words)

198

In [ ]:
from nltk.tokenize import word_tokenize
def remove_stop_words(text):
  tokens=word_tokenize(text)
  return ' '.join([word for word in tokens if word not in stop_words])

In [ ]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
df['cleaned_data']=df['cleaned_data'].apply(remove_stop_words)

In [ ]:
df.sample(3)

,Review,sentiment,cleaned_data
196,im very disappointed with the service,negative,im disappointed service
5699,i absolutely loved this movie 😊,positive,absolutely loved movie smiling_face_with_smili...
80,i absolutely loved this movie 😊,positive,absolutely loved movie smiling_face_with_smili...


In [ ]:
df['cleaned_data']=df['cleaned_data'].apply(word_tokenize)

In [ ]:
df

,Review,sentiment,cleaned_data
0,i absolutely loved this movie 😊 😍,positive,"[absolutely, loved, movie, smiling_face_with_s..."
1,terrible experience would not recommend,negative,"[terrible, experience, would, recommend]"
2,i absolutely loved this movie 😊 😍,positive,"[absolutely, loved, movie, smiling_face_with_s..."
3,had a great time with friends everything was p...,positive,"[great, time, friends, everything, perfect, la..."
4,i absolutely loved this movie 😊,positive,"[absolutely, loved, movie, smiling_face_with_s..."
...,...,...,...
5995,what a fantastic experience will come again,positive,"[fantastic, experience, come]"
5996,worst product ever total waste of money,negative,"[worst, product, ever, total, waste, money]"
5997,had a great time with friends everything was p...,positive,"[great, time, friends, everything, perfect]"
5998,the service was excellent highly recommend,positive,"[service, excellent, highly, recommend]"


In [ ]:
#stemming: reduce word forms and noise ---> words=> root words
stemmer=PorterStemmer()
def stem_words(text):
  return[stemmer.stem(word) for word in text]

In [ ]:
df['cleaned_data'].apply(stem_words)

,cleaned_data
0,"[absolut, love, movi, smiling_face_with_smilin..."
1,"[terribl, experi, would, recommend]"
2,"[absolut, love, movi, smiling_face_with_smilin..."
3,"[great, time, friend, everyth, perfect, laught..."
4,"[absolut, love, movi, smiling_face_with_smilin..."
...,...
5995,"[fantast, experi, come]"
5996,"[worst, product, ever, total, wast, money]"
5997,"[great, time, friend, everyth, perfect]"
5998,"[servic, excel, highli, recommend]"


#Lemmetization
  convert words to dict form
  eg better --> good
     ....ing ---> ....
     ----less --> ----
     ---->keep real words


In [ ]:
lemmetizer=WordNetLemmatizer()
def lemm_words(text):
  return[lemmetizer.lemmatize(word) for word in text]


In [ ]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
df['cleaned_data']=df['cleaned_data'].apply(lemm_words)

Feature Extraction

1.BOW


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
df['BOW_feedback']=df['cleaned_data'].apply(lambda x:' '.join(x))
vectorizer=CountVectorizer()
bow=vectorizer.fit_transform(df['BOW_feedback'])
print(bow)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 27457 stored elements and shape (6000, 39)>
  Coords	Values
  (0, 0)	1
  (0, 18)	1
  (0, 21)	1
  (0, 30)	1
  (0, 29)	1
  (0, 9)	1
  (1, 32)	1
  (1, 8)	1
  (1, 38)	1
  (1, 26)	1
  (2, 0)	1
  (2, 18)	1
  (2, 21)	1
  (2, 30)	1
  (2, 29)	1
  (2, 9)	1
  (3, 12)	1
  (3, 33)	1
  (3, 11)	1
  (3, 6)	1
  (3, 22)	1
  (3, 16)	1
  (3, 17)	1
  (4, 0)	1
  (4, 18)	1
  :	:
  (5994, 1)	1
  (5995, 8)	1
  (5995, 10)	1
  (5995, 1)	1
  (5996, 24)	1
  (5996, 36)	1
  (5996, 4)	1
  (5996, 34)	1
  (5996, 35)	1
  (5996, 20)	1
  (5997, 12)	1
  (5997, 33)	1
  (5997, 11)	1
  (5997, 6)	1
  (5997, 22)	1
  (5998, 26)	1
  (5998, 28)	1
  (5998, 7)	1
  (5998, 14)	1
  (5999, 24)	1
  (5999, 36)	1
  (5999, 4)	1
  (5999, 34)	1
  (5999, 35)	1
  (5999, 20)	1


In [ ]:
bow[0].toarray()

array([[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1,
        0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])

In [ ]:
#TFIDF
from sklearn.feature_extraction.text import  TfidfVectorizer
vectorizer=TfidfVectorizer()
tfidf=vectorizer.fit_transform(df['BOW_feedback'])
print(tfidf)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 27457 stored elements and shape (6000, 39)>
  Coords	Values
  (0, 0)	0.4253702788450976
  (0, 18)	0.4253702788450976
  (0, 21)	0.4253702788450976
  (0, 30)	0.4253702788450976
  (0, 29)	0.37164533059153027
  (0, 9)	0.37164533059153027
  (1, 32)	0.5539885657661933
  (1, 8)	0.44215900642417705
  (1, 38)	0.5539885657661933
  (1, 26)	0.4366792312883873
  (2, 0)	0.4253702788450976
  (2, 18)	0.4253702788450976
  (2, 21)	0.4253702788450976
  (2, 30)	0.4253702788450976
  (2, 29)	0.37164533059153027
  (2, 9)	0.37164533059153027
  (3, 12)	0.4015576307718503
  (3, 33)	0.31991348270010334
  (3, 11)	0.4015576307718503
  (3, 6)	0.4015576307718503
  (3, 22)	0.4015576307718503
  (3, 16)	0.3554301902359198
  (3, 17)	0.3554301902359198
  (4, 0)	0.5
  (4, 18)	0.5
  :	:
  (5994, 1)	0.6182862067221117
  (5995, 8)	0.4852260639685017
  (5995, 10)	0.6182862067221117
  (5995, 1)	0.6182862067221117
  (5996, 24)	0.3314043613957658
  (5996, 36)	0.421941

In [ ]:
tfidf[1].toarray()

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.44215901, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.43667923, 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.55398857, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.55398857]])

In [ ]:
tfidf.shape

(6000, 39)

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 74.7 MB/s eta 0:00:00


In [ ]:
from gensim.models import Word2Vec

In [ ]:
sent=df['BOW_feedback'].apply(lambda x: x.split()).tolist()
print(sent)

[['absolutely', 'loved', 'movie', 'smiling_face_with_smiling_eyes', 'smiling_face_with_heart-eyes'], ['terrible', 'experience', 'would', 'recommend'], ['absolutely', 'loved', 'movie', 'smiling_face_with_smiling_eyes', 'smiling_face_with_heart-eyes'], ['great', 'time', 'friend', 'everything', 'perfect', 'laught', 'loud'], ['absolutely', 'loved', 'movie', 'smiling_face_with_smiling_eyes'], ['service', 'excellent', 'highly', 'recommend'], ['great', 'time', 'friend', 'everything', 'perfect'], ['terrible', 'experience', 'would', 'recommend', 'laught', 'loud'], ['hated', 'every', 'minute', 'enraged_face'], ['superb', 'product', 'quality', 'satisfied'], ['terrible', 'experience', 'would', 'recommend'], ['hated', 'every', 'minute', 'enraged_face'], ['hated', 'every', 'minute', 'enraged_face'], ['worth', 'time', 'price'], ['superb', 'product', 'quality', 'satisfied'], ['superb', 'product', 'quality', 'satisfied'], ['worth', 'time', 'price'], ['worth', 'time', 'price'], ['absolutely', 'loved', '

In [ ]:
model=Word2Vec(sent,min_count=1,vector_size=100,window=3,workers=4)
model.wv['would']

array([ 6.25796393e-02,  8.13859329e-02,  8.96032527e-02,  2.12501481e-01,
       -6.96019679e-02, -1.30322024e-01,  2.29928911e-01,  2.47221112e-01,
       -4.57234345e-02, -3.46897721e-01,  1.19848944e-01, -1.21888816e-01,
        1.29605472e-01,  1.59815818e-01,  1.07007198e-01, -6.80041984e-02,
        2.96743035e-01,  2.84330957e-02, -1.56476527e-01, -2.68950105e-01,
        8.81628469e-02,  2.41965465e-02,  1.18156344e-01, -4.09101024e-02,
       -1.38963655e-01,  6.54092878e-02, -1.44219965e-01,  8.60228986e-02,
       -1.05007418e-01,  1.98611677e-01,  1.53789967e-01, -1.64294019e-01,
        6.93711489e-02, -3.58805329e-01, -7.70112202e-02,  1.98133990e-01,
        4.47365381e-02, -5.74503131e-02, -1.15721166e-01,  2.00656541e-02,
        1.10053435e-01, -1.38062075e-01, -1.73103306e-02,  1.45349517e-01,
        1.80636659e-01, -9.89441499e-02, -9.61004347e-02, -2.19552442e-01,
        2.28239685e-01,  8.93701240e-02,  1.35783060e-02, -1.94896892e-01,
       -1.58517018e-01, -

In [ ]:
from scipy.sparse import csr_matrix
x=tfidf
feature_list=[x[i].toarray().flatten() for i in range(x.shape[0])]
feature_list=np.array(feature_list)
feature_list
################ sparse matrix
sparse_list =[x[i] for i in range(x.shape[0])]
sparse_list

[<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 6 stored elements and shape (1, 39)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 4 stored elements and shape (1, 39)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 6 stored elements and shape (1, 39)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 7 stored elements and shape (1, 39)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 4 stored elements and shape (1, 39)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 4 stored elements and shape (1, 39)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 5 stored elements and shape (1, 39)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 6 stored elements and shape (1, 39)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 4 stored elements and shape (1, 39)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 4 stored elements 

In [ ]:
from scipy.sparse import save_npz
save_npz('tfidf_matrix.npz',x)

In [ ]:
from scipy.sparse import csr_matrix,load_npz
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.preprocessing import LabelEncoder

# Encode the target variable
le = LabelEncoder()
df['sentiment_encoded'] = le.fit_transform(df['sentiment'])

x_train, x_test, y_train, y_test = train_test_split(tfidf, df['sentiment_encoded'], test_size=0.2, random_state=42)

In [ ]:
df

,Review,sentiment,cleaned_data,BOW_feedback,sentiment_encoded
0,i absolutely loved this movie 😊 😍,positive,"[absolutely, loved, movie, smiling_face_with_s...",absolutely loved movie smiling_face_with_smili...,1
1,terrible experience would not recommend,negative,"[terrible, experience, would, recommend]",terrible experience would recommend,0
2,i absolutely loved this movie 😊 😍,positive,"[absolutely, loved, movie, smiling_face_with_s...",absolutely loved movie smiling_face_with_smili...,1
3,had a great time with friends everything was p...,positive,"[great, time, friend, everything, perfect, lau...",great time friend everything perfect laught loud,1
4,i absolutely loved this movie 😊,positive,"[absolutely, loved, movie, smiling_face_with_s...",absolutely loved movie smiling_face_with_smili...,1
...,...,...,...,...,...
5995,what a fantastic experience will come again,positive,"[fantastic, experience, come]",fantastic experience come,1
5996,worst product ever total waste of money,negative,"[worst, product, ever, total, waste, money]",worst product ever total waste money,0
5997,had a great time with friends everything was p...,positive,"[great, time, friend, everything, perfect]",great time friend everything perfect,1
5998,the service was excellent highly recommend,positive,"[service, excellent, highly, recommend]",service excellent highly recommend,1


In [ ]:
model=LogisticRegression()
model.fit(x_train,y_train)
y_pred=model.predict(x_test)
print(accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       571
           1       1.00      1.00      1.00       629

    accuracy                           1.00      1200
   macro avg       1.00      1.00      1.00      1200
weighted avg       1.00      1.00      1.00      1200



In [ ]:
#Evaluate model
y_pred=model.predict(x_train)
print(accuracy_score(y_train,y_pred))
print(classification_report(y_train,y_pred))

1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2470
           1       1.00      1.00      1.00      2330

    accuracy                           1.00      4800
   macro avg       1.00      1.00      1.00      4800
weighted avg       1.00      1.00      1.00      4800



In [ ]:
def clean_text(text):
    # 1. Lowercasing
    text = text.lower()
    # 2. Remove HTML tags
    text = BeautifulSoup(text, 'html.parser').get_text()
    # 3. Remove URLs
    text = re.sub(r'http\S+', '', text)
    # 4. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # 5. Expand chat words
    text = ' '.join([word_data[word] if word in word_data else word for word in text.split()])
    # 6. Spelling correction (can be resource intensive, consider carefully for production)
    text = str(TextBlob(text).correct())
    # 7. Handle emojis
    text = emoji.demojize(text, delimiters=(" ", " "))
    # 8. Remove stop words and tokenize
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    # 9. Stemming
    tokens = [stemmer.stem(word) for word in tokens]
    # 10. Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    # 11. Join back to string for vectorizer
    return ' '.join(tokens)

def predict_feedback():
    user_feedback = input("Enter your feedback: ")
    # 1. Clean user input
    cleaned_feedback = clean_text(user_feedback)
    # 2. Vectorize using already fitted vectorizer
    user_vector = vectorizer.transform([cleaned_feedback]) # Changed from tfidf.transform to vectorizer.transform
    # 3. Predict
    prediction = model.predict(user_vector)[0]
    # 4. Display result
    # Map the numerical prediction back to the original sentiment labels
    predicted_sentiment = le.inverse_transform([prediction])[0]
    print("Feedback:", user_feedback)
    print("Prediction:", predicted_sentiment)

In [ ]:
predict_feedback()

Enter your feedback: movie is good
Feedback: movie is good
Prediction: negative
